# Milestone 3: LLM Prompt Engineering and Reasoning

This notebook investigates how different prompting strategies affect an LLM's analysis of synthetic patient similarity.

The same query patient and retrieved similar patients are used accorss all experiments so that differences in the generated responses can be attributed primarily to the prompting strategy.

Four prompting methods are evaluated:

1. Zero-shot prompting
2. Few-shot / in-context learning
3. Chain-of-Thought-style structured reasoning
4. Tree-of-Thought-style multi-perspective reasoning

All patient records used in these experiments originate from the synthetic Synthea COVID-19 dataset.

## 1. Project Setup

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

### Import Preprocessing Functions and Prompt Templates

In [2]:
from src.preprocessing import load_patient_master
from src.similarity import (
    build_feature_matrix,
    find_similar_patients
)
from src.patient_summary import generate_similarity_context

from src.prompt_templates import (
    zero_shot_prompt,
    few_shot_prompt,
    chain_of_thought_prompt,
    tree_of_thought_prompt,
    similarity_classification_prompt
)

from src.llm_utils import call_llm

### Define Data Path and Build Patient Context

In [3]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

patient_df = load_patient_master(PROCESSED_DATA_DIR)

patient_df, feature_matrix, artifacts = build_feature_matrix(
    patient_df
)

query_index = 0

similar_patients = find_similar_patients(
    patient_df,
    feature_matrix,
    query_index=query_index,
    k=3,
)

patient_context = generate_similarity_context(
    patient_df.iloc[query_index],
    similar_patients,
)

print(patient_context)

QUERY PATIENT
Patient Profile

Age: 8
Gender: M
Race: white
Ethnicity: nonhispanic

Encounter Count: 5

Conditions:
- Otitis media
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- Amoxicillin 250 MG Oral Capsule
- Acetaminophen 160 MG Chewable Tablet

SIMILAR PATIENT 1
Patient Profile

Age: 9
Gender: M
Race: black
Ethnicity: nonhispanic

Encounter Count: 5

Conditions:
- Otitis media
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- Amoxicillin 250 MG Oral Capsule
- Ibuprofen 100 MG Oral Tablet

SIMILAR PATIENT 2
Patient Profile

Age: 7
Gender: M
Race: white
Ethnicity: nonhispanic

Encounter Count: 5

Conditions:
- Otitis media
- Cough (finding)
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- None recorded

SIMILAR PATIENT 3
Patient Profile

Age: 9
Gender: M
Race: asian
Ethnicity: nonhispanic

Encounter Count: 4

Conditions:
- Otitis media
- Cough (finding)
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- None recorded

### Zero-Shot Context

In [4]:
zero_shot = zero_shot_prompt(patient_context)

print(zero_shot)


You are analyzing synthetic patient records for an educational healthcare AI project.

Compare the query patient with the three retrieved patients.

Explain:
- the major clinical similarities,
- the important differences,
- and why these patients may have been retrieved as similar.

Do not make diagnoses or treatment recommendations.
Base your answer only on the supplied information.

The similarity retrieval system used the following features:
- TF-IDF representation of recorded clinical conditions
- Age
- Gender
- Encounter count

Race, ethnicity, and medication history were NOT used to calculate
the similarity scores. You may discuss these attributes as observed
similarities or differences, but do not claim that they caused a
patient to be retrieved.

PATIENT RECORDS
----------------
QUERY PATIENT
Patient Profile

Age: 8
Gender: M
Race: white
Ethnicity: nonhispanic

Encounter Count: 5

Conditions:
- Otitis media
- Fever (finding)
- Suspected COVID-19
- COVID-19

Medications:
- Amox

### Few-Shot Context

In [5]:
few_shot = few_shot_prompt(patient_context)

print(few_shot)


You are analyzing synthetic patient records for an educational healthcare AI project.

Below are examples showing how patient similarity should be analyzed.

EXAMPLE 1

Query Patient:
Age: 65
Gender: F
Conditions: Hypertension, Diabetes

Similar Patient:
Age: 67
Gender: F
Conditions: Hypertension, Diabetes, Hyperlipidemia

Analysis:
These patients are similar because they are close in age 
and share hypertension and diabetes. The similar patient 
also has hyperlipidemia, which is an important clinical 
difference.

EXAMPLE 2

Query Patient:
Age: 32
Gender: M
Conditions: Asthma, Cough

Similar Patient:
Age: 35
Gender: M
Conditions: Asthma, Fever

Analysis:
Both patients have asthma and are similar in age and gender. 
However, their associated symptoms differ: the query patient 
has cough while the comparison patient has fever.

Now apply the same comparison approach to the following synthetic patient recordds.

Identify:
1. Major clinical similarities
2. Important differences
3. Why th

### Chain-of-Thought-style Context

In [6]:
cot_prompt = chain_of_thought_prompt(patient_context)

print(cot_prompt)


You are analyzing synthetic patient records for an educational healthcare AI project.

Analyze the query patient and retrieved patients using the following structured reasoning process.

Step 1: Compare their demographics, ecpecially age and gender.

Step 2: Compare their clinical conditions and identify conditions shared with the query patient.

Step 3: Compare their medication histories and identify
important similarities or differences.

Step 4: Compare healthcare utilization using encounter counts.

Step 5: Based on the evidence above, provide a concise explanation
of why the retrieved patients are clinicalLy similar to the query patient.

Clearly report the result of each step. Use only the
information supplied in the patient records. Do not make
new diagnoses or treatment recommendations.

The similarity retrieval system used the following features:
- TF-IDF representation of recorded clinical conditions
- Age
- Gender
- Encounter count

Race, ethnicity, and medication history w

### Tree-of-Thought-style Context

In [7]:
tot_prompt = tree_of_thought_prompt(patient_context)

print(tot_prompt)


You are analyzing synthetic patient records for an educational healthcare AI project.

Evaluate the similarity between the query patient and the 
retrieved patients from three separate perspectives.

Perspective A - Clinical Conditions
Compare diagnoses and symptoms. Identify important shared conditions and meaningful differences.

Perspective B - Demographics
Compare age and gender and determine how strongly these characteristics support similarity.

Perspective C - Treatment and Healthcare Utilization
Compare medication histories and encounter counts.

For each perspective:
- identify the strongest evidence for similarity,
- identify important differences,
- and briefly assess how informative that perspective is.

After evaluating all three perspectives, synthesize them 
into a final explanationof why the retrieved patient are 
similar to the query patient.

Base the analysis only on the supplied synthetic records.
Do not make diagnoses or treatment recommendations.

The similarity 

## 2. Define Model and Perform Prompting Experiments

In [9]:
MODEL = "gpt-5.6"

## Experiment 1: Zero-Shot Prompting

Zero-shot prompting provides the LLM with the patient records and task instructions without demonstrating how the task should be completed.

This experiment serves as the baseline for evaluating whether additional examples or structured reasoning instructions improve the analysis of patient similarity.

In [11]:
zero_responses = call_llm(
    zero_shot,
    model=MODEL
)

In [12]:
print(zero_responses)

### Major clinical similarities

- All four patients are boys of similar age (7–9 years).
- Every patient has the same core recorded conditions:
  - Otitis media
  - Fever
  - Suspected COVID-19
  - COVID-19
- The query patient and Similar Patient 1 have an exact match in their condition lists.
- Similar Patients 2 and 3 share all of the query patient’s conditions but also have cough recorded.

### Important differences

- **Age:** The query patient is 8; Similar Patient 1 and 3 are 9, while Similar Patient 2 is 7.
- **Encounter count:** The query patient and Similar Patients 1 and 2 each have five encounters. Similar Patient 3 has four.
- **Medications:**
  - The query patient has amoxicillin and acetaminophen recorded.
  - Similar Patient 1 also has amoxicillin, but has ibuprofen rather than acetaminophen.
  - Similar Patients 2 and 3 have no medications recorded.
- **Race:** The query patient is recorded as white; the retrieved patients are black, white, and Asian, respectively. All

### Zero-Shot Interpretation

The zero-shot model accurately identified the major clinical similarities and differences among the retrieved patients. it correctly emphasized the strong overlap in COVID-19 related conditions and recognized age, gender, and encounter count as additional retrieval features.

After explicitly providing information about the retrieval algorithm, the model also correctly distinguished between features used to calculate similarity and attributes that were available only for descriptive comparison. In paarticular, it did not attribute retrieval to race, ethnicity, or medication history.

The response was concise and clinically coherent, but it primarily summarized the retrieved patients collectively rather than evaluating each retrieved patient in a consistent individual structure.

## Experiment 2: Few-Shot / In-Context Prompting

In [13]:
few_shot_responses = call_llm(
    few_shot,
    model=MODEL
)

print(few_shot_responses)

### Similar Patient 1

**Major similarities**
- Nearly identical age (8 vs. 9), and both are male.
- Exact match across all recorded conditions: otitis media, fever, suspected COVID-19, and COVID-19.
- Both have 5 encounters.
- Both are recorded as non-Hispanic and received amoxicillin.

**Important differences**
- Recorded race differs: query patient is white; similar patient is black.
- The query patient has acetaminophen recorded, while Similar Patient 1 has ibuprofen.

**Why retrieved as similar**
- The condition profiles are identical, and age, gender, and encounter count are also closely matched. Race and medications were not retrieval features.

---

### Similar Patient 2

**Major similarities**
- Close in age (8 vs. 7), with the same gender.
- Shares all four query conditions: otitis media, fever, suspected COVID-19, and COVID-19.
- Both have 5 encounters.
- Race and ethnicity are also the same.

**Important differences**
- Similar Patient 2 additionally has cough recorded.
- T

### Few-Shot / In-Context Learning Interpretation

The few-shot response remained factually consistent with the patient records but produced a more systematic comparison than the zero-shot baseline.

After observing example comparisons in the prompt, the model applied a repeated structure to each retrieved patient: major similarities, important differences, and an explanation of why the patient was retrieved. This made the relationship between individual patient characteristics and the retrieval result easier to trace.

The few-shot examples therefore appeared to influence the organization and consistency of the response more strongly than its factual content. Both methods identified similar clinical evidence, but few-shot prompting produced a more structured patient-by-patient analysis.

## Experiment 3: Chain-of-Thought Prompting

In [14]:
cot_responses = call_llm(
    cot_prompt,
    model=MODEL
)

print(cot_responses)

### Step 1: Demographics

- **Query patient:** 8-year-old male.
- **Similar Patient 1:** 9-year-old male; one year older.
- **Similar Patient 2:** 7-year-old male; one year younger.
- **Similar Patient 3:** 9-year-old male; one year older.

All are male children within one year of the query patient’s age. Patients 1 and 3 differ in recorded race from the query patient, while Patient 2 has the same recorded race. All are recorded as non-Hispanic. Race and ethnicity were not used for retrieval.

### Step 2: Clinical Conditions

All three retrieved patients share the query patient’s complete set of recorded conditions:

- Otitis media
- Fever
- Suspected COVID-19
- COVID-19

Patients 2 and 3 additionally have **cough**, which is not recorded for the query patient. Patient 1 has an exact condition-list match.

### Step 3: Medication Histories

- **Patient 1:** Shares **amoxicillin 250 mg** with the query patient. The query patient has acetaminophen, whereas Patient 1 has ibuprofen.
- **Pat

### Chain-of-Thought Interpretation

The Chain-of-Thought-style prompt produced a sequential and structured analysis across demographics, clinical conditions, medication history, and healthcare utilization before generating an overall similarity assessment.

Compared with zero-shot and few-shot prompting, this approach made the analytical process more explicit. Each feature catagory eas examined separately, making it easier to verify whether the final explanation was supported by the patient records.

The model correctly identified the complete overlap in the query patient's core conditions, the close age and encounter counts, and the differences in medication history. It also maintained retrieval faithfulness by distinguishing medication history, race, and ethnicity from the features actually used to calculate similarity.

The main benefit of this approach was therefore transparency and completeness rather than discovery of substantially different clinical facts.

## Experiment 4: Tree-of-Thought Prompting

In [15]:
tot_responses = call_llm(
    tot_prompt,
    model=MODEL
)

print(tot_responses)

## Perspective A — Clinical Conditions

**Strongest evidence for similarity**
- All three retrieved patients share the query patient’s four recorded conditions: **otitis media, fever, suspected COVID-19, and COVID-19**.
- Similar Patient 1 has an **exact condition match**.
- Similar Patients 2 and 3 have the same core condition set plus **cough**.

**Important differences**
- Cough is recorded for Similar Patients 2 and 3 but not for the query patient.
- There are no other condition differences in the supplied records.

**How informative this is**
- This is the **most informative perspective**. The exact or near-exact overlap in conditions strongly supports similarity, particularly because condition TF-IDF was used by the retrieval system.

## Perspective B — Demographics

**Strongest evidence for similarity**
- All patients are **male**.
- Their ages are tightly clustered: the query patient is 8, Similar Patient 2 is 7, and Similar Patients 1 and 3 are 9.
- All are also recorded as no

### Tree-of-Thought Interpretation

The Tree-of-Thought-style prompt produced the most explicitly multi-perspective analysis. Instead of following one sequential reasoning path, the response independently evaluated clinical conditions, demographics, and treatment/healthcare utilization before synthesizing the evidence.

A notable difference was that the model assessed how informative each perspective was. It identified clinical conditions as the strongest source of similarity, while treating age and gender as supporting evidence and medication history as secondary observational information because medications were not used by the retrieval algorithm.

This evidence prioritization made the final explanation more nuanced than the other prompting strategies. The approach also maintained factual consistency and correctly distinguished retrieval features from descriptive patient attributes.

The primary trade-off was verbosity: the Tree-of-Thought response provided the richest explanation but required substantially more text than the zero-shot response.

## Comparison of Prompting Strategies

All four prompting methods successfully identified the major clinical similarities among the retrieved synthetic patients, including shared COVID-19-related conditions, close ages, identical gender, and similar encounter counts. No major factual hallucinations were observed in the final experimental outputs.

However, prompt design noticeably affected the organization and depth of the analysis.

- **Zero-shot prompting** produced the most concise overall explanation and served as an effective baseline.
- **Few-shot prompting** adopted the structure demonstrated in the in-context examples, resulting in a consistent patient-by-patient comparison.
- **Chain-of-Thought-style prompting** decomposed the task into sequential feature categories, improving transparency and completeness.
- **Tree-of-Thought-style prompting** considered multiple perspectives independently and explicitly evaluated their relative importance before producing a final synthesis.

These results suggest that more structured prompting did not necessarily reveal new patient facts, but it changed how systematically the model organized, justified, and prioritized the evidence used to explain patient similarity.

## 3. Save the Results

In [17]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "METHOD": [
            "Zero-shot",
            "Few-shot",
            "Chain-of-Thought",
            "Tree-of-Thought",
        ],
        "PROMPT": [
            zero_shot,
            few_shot,
            cot_prompt,
            tot_prompt,
        ],
        "RESPONSE": [
            zero_responses,
            few_shot_responses,
            cot_responses,
            tot_responses,
        ],
    }
)

results_df.to_csv(
    PROJECT_ROOT / "outputs" / "prompt_results.csv",
    index=False,
)

## Experiment 5: Few-Shot Patient Similarity Classification

The previous experiments treated patient similarity explanation as a
text-generation task. This experiment evaluates whether an LLM can
also perform a downstream classification task.

Using few-shot in-context examples and structured reasoning
instructions, the model classifies each retrieved patient into one of
three similarity categories:

- **HIGH**
- **MODERATE**
- **LOW**

The classification is based only on the features used by the retrieval
system: clinical conditions, age, gender, and encounter count.

This experiment demonstrates how the same patient representations can
support a different downstream LLM task while combining in-context
learning with structured reasoning.

In [18]:
classification_prompt = similarity_classification_prompt(patient_context)

In [19]:
classification_response = call_llm(
    classification_prompt,
    model=MODEL
)

print(classification_response)

### Similar Patient 1 — HIGH
Shares all four recorded conditions with the query patient. The patient is the same gender, only one year older, and has the same encounter count (5).

### Similar Patient 2 — HIGH
Shares all four query conditions, with one additional condition (cough). The patient is the same gender, only one year younger, and has the same encounter count (5).

### Similar Patient 3 — HIGH
Shares all four query conditions, with one additional condition (cough). The patient is the same gender, only one year older, and has a very similar encounter count (4 versus 5).


### Classification Results

The LLM classified all three retrieved patients as having **HIGH**
similarity to the query patient.

These classifications are consistent with the cosine similarity scores
produced by the retrieval system, which were greater than 0.99 for all
three patients.

The model's explanations also aligned with the features used by the
retrieval algorithm. It emphasized:

- Complete overlap in the query patient's four core conditions
- Similar patient ages
- Identical gender
- Identical or nearly identical encounter counts

The LLM did not use race, ethnicity, or medication history as evidence
for its classifications, consistent with the instructions provided in
the prompt.

This experiment demonstrates a downstream classification task combining
few-shot in-context learning with structured evidence evaluation. The
LLM's qualitative HIGH classifications were consistent with the
numerical similarity scores generated by the retrieval system.

## Milestone 3 Summary

This milestone evaluated multiple LLM prompting strategies for analyzing
similarity among synthetic patient records.

Five experiments were conducted:

1. **Zero-shot prompting** — generated a direct comparison without examples.
2. **Few-shot prompting** — used in-context demonstrations to guide the
   structure of patient comparisons.
3. **Chain-of-Thought-style prompting** — decomposed the analysis into
   sequential evidence categories.
4. **Tree-of-Thought-style prompting** — evaluated multiple perspectives
   independently before synthesizing the evidence.
5. **Few-shot similarity classification** — combined in-context examples
   and structured evidence evaluation to classify patient similarity as
   HIGH, MODERATE, or LOW.

Across the explanation experiments, the underlying clinical conclusions
were largely consistent, but prompt design substantially affected the
organization, transparency, and prioritization of evidence.

Zero-shot prompting provided the most concise analysis. Few-shot
prompting produced a more consistent patient-by-patient structure.
Chain-of-Thought-style prompting made the analytical sequence explicit,
while Tree-of-Thought-style prompting provided the clearest prioritization
of different sources of similarity evidence.

The downstream classification experiment further demonstrated that an
LLM could use the same synthetic patient context for a different task.
All three retrieved patients were classified as HIGH similarity, consistent
with their cosine similarity scores above 0.99.